# Module 13 Lab - Building ML Pipelines**Objective:** To understand the importance of `scikit-learn` **Pipelines** for creating robust, reproducible, and professional machine learning workflows.**In this lab, you will refactor code from a previous lab into a clean, professional `Pipeline` object.**

## Part 1: Why Use Pipelines?**Concept:** As you've seen, a typical ML workflow involves multiple steps: loading data, cleaning it, splitting it, preprocessing features (scaling, encoding), and finally, training a model. Managing all these steps separately can be messy and error-prone.**Data Leakage:** A major risk of manual preprocessing is **data leakage**. This happens when information from the test set accidentally "leaks" into the training process. For example, if you calculate the mean for scaling using the *entire* dataset before splitting, the model has already "seen" the test data, leading to overly optimistic performance estimates.**A `scikit-learn` Pipeline solves these problems by:**1.  **Encapsulating** all workflow steps into a single object.2.  **Preventing Data Leakage:** It ensures that preprocessing steps are fitted *only* on the training data during cross-validation or when calling `.fit()`.3.  **Improving Reproducibility:** The entire workflow is saved as one object, making it easy to reuse and deploy.

## Part 2: The "Manual" Way (What We Did Before)Let's revisit the Titanic dataset and the steps we took to prepare the data and train a model. This code should look familiar. Notice how many separate objects and steps there are.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


# Load Titanic dataset
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

df = pd.read_csv(url)


# Clean and engineer features
df["Age"].fillna(df["Age"].median(), inplace=True)

df["Embarked"].fillna(
    df["Embarked"].mode()[0],
    inplace=True,
)

df.drop("Cabin", axis=1, inplace=True)


# Separate features and target
X = df.drop(
    ["Survived", "Name", "Ticket", "PassengerId"],
    axis=1,
)

y = df["Survived"]


# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)


# Define feature groups
numeric_features = [
    "Age",
    "Fare",
    "SibSp",
    "Parch",
]

categorical_features = [
    "Pclass",
    "Sex",
    "Embarked",
]


# Scale numerical features
scaler = StandardScaler()

X_train_scaled_num = scaler.fit_transform(
    X_train[numeric_features]
)

# Use learned scaling values on test data
X_test_scaled_num = scaler.transform(
    X_test[numeric_features]
)


# Encode categorical features
encoder = OneHotEncoder(
    handle_unknown="ignore"
)

X_train_encoded_cat = encoder.fit_transform(
    X_train[categorical_features]
)

# Use learned encoding on test data
X_test_encoded_cat = encoder.transform(
    X_test[categorical_features]
)


# Combine processed features
X_train_processed = np.hstack(
    (
        X_train_scaled_num,
        X_train_encoded_cat.toarray(),
    )
)

X_test_processed = np.hstack(
    (
        X_test_scaled_num,
        X_test_encoded_cat.toarray(),
    )
)


# Train Random Forest model
model = RandomForestClassifier(
    random_state=42
)

model.fit(
    X_train_processed,
    y_train,
)


# Predict and evaluate
y_pred = model.predict(
    X_test_processed
)

accuracy = accuracy_score(
    y_test,
    y_pred,
)

print(f"Accuracy (Manual Method): {accuracy:.2%}")

/tmp/ipykernel_573/1066676012.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Age"].fillna(df["Age"].median(), inplace=True)
/tmp/ipykernel_573/1066676012.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', tr

Accuracy (Manual Method): 82.68%


## Part 3: The "Pipeline" WayNow, let's do the exact same thing but encapsulate all the preprocessing steps into a single `Pipeline`.**Your Task:** Use `make_pipeline` and `make_column_transformer` to build a complete workflow. This is the modern, professional way to build models in `scikit-learn`.

In [2]:
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split


# Load Titanic dataset
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

df = pd.read_csv(url)


# Remove unnecessary columns
df.drop(
    ["Cabin", "Name", "Ticket", "PassengerId"],
    axis=1,
    inplace=True,
)


# Separate features and target
X = df.drop("Survived", axis=1)
y = df["Survived"]


# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)


# Pipeline for numerical features
numeric_transformer = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
)


# Pipeline for categorical features
categorical_transformer = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"),
)


# Apply different preprocessing to each column type
preprocessor = make_column_transformer(
    (
        numeric_transformer,
        ["Age", "Fare", "SibSp", "Parch"],
    ),
    (
        categorical_transformer,
        ["Pclass", "Sex", "Embarked"],
    ),
)


# Combine preprocessing and model
final_pipeline = make_pipeline(
    preprocessor,
    RandomForestClassifier(random_state=42),
)


# Train and evaluate the full workflow
final_pipeline.fit(
    X_train,
    y_train,
)

y_pred_pipeline = final_pipeline.predict(
    X_test,
)

accuracy = accuracy_score(
    y_test,
    y_pred_pipeline,
)

print(f"Accuracy (Pipeline Method): {accuracy:.2%}")

Accuracy (Pipeline Method): 82.68%


## 📝 Reflective Knowledge Check**Instructions:** Answer the following questions in this markdown cell.1.  **Code Comparison:** Look at the "Manual Way" versus the "Pipeline Way". What are the three biggest advantages you see in using the Pipeline approach?2.  **Data Leakage Explained:** In the manual code, we used `scaler.fit_transform()` on the training data but only `scaler.transform()` on the test data. Why was this distinction crucial? How does the Pipeline automatically handle this for you?3.  **Extending the Pipeline:** Imagine you wanted to add a PCA step to reduce dimensionality *after* scaling and encoding but *before* the RandomForestClassifier. How would you modify your `final_pipeline` object to include this step? (You don't need to write the full code, just describe where you would add `PCA()`.)4.  **Real-World Value:** You are handing your model over to another team to deploy into a web application. Why is giving them the single `final_pipeline` object much safer and more reliable than giving them the 5 separate objects (`scaler`, `encoder`, `model`, etc.) from the manual approach?

**1. Code Comparison:**

The three biggest advantages of using the Pipeline approach over the manual approach are as follows:

-- **Preventing data leakage** by enforcing the correct order of steps and training solely on training data.

-- **Improving reproducibility** by storing preprocessing and the model together in one object.

-- **Simplifying maintenance** since the entire workflow can be reused with a single pipeline object.

**2. Data Leakage Explained:**

we used `scaler.fit_transform()` on the training data in the manual approach because the scaler needed to learn the training data's mean and standard deviation.

We used only `scaler.transform()` on the test data because the test set should not influence the preprocessing.

This lab shows that The Pipeline approach automatically handles this step by fitting preprocessing steps only on the training data when `.fit()` is called and applying the learned transformations to new data during `.predict()`.

3. **Extending the Pipeline**

 here's a snippet of the code modification needed to add PCA: from sklearn.decomposition import PCA

```python
final_pipeline = make_pipeline(
    preprocessor,
    PCA(n_components=0.95),
    RandomForestClassifier(random_state=42),
)
```
as the code above shows, To add PCA, you would need to insert it between the `preprocessor` and `RandomForestClassifier` so dimensionality reduction occurs after scaling and encoding.


**4. Real-World Value:**

Giving another team the single `final_pipeline` object is safer because it contains the entire workflow in one place, including preprocessing and the trained model.

With separate objects, there is a business risk that someone could mess up the  preprocessing steps or use them in the wrong order. Also, as emphasized in the lab, the pipeline approach guarantees that new data is processed exactly the same way as the training data, making deployment more reliable.